# Задание №1

In [1]:
import re

In [2]:
def check_car_id(car_id: str) -> str:
    pattern = r'[AАВBЕEКKМНОOРPСCТУХX]\d{3}[AАВBЕEКKМНОOРPСCТУХX]{2}\d{2,3}'
    if re.fullmatch(pattern, car_id):
        if car_id[-3].isnumeric():
            region = car_id[-3:]
        else:
            region = car_id[-2:]
        return f'Номер {car_id[:6]} валиден. Регион: {region}'
    else:
        return 'Номер не валиден'

In [3]:
check_car_id('А222ВС96')

'Номер А222ВС валиден. Регион: 96'

In [4]:
check_car_id('А123ВЕ777')

'Номер А123ВЕ валиден. Регион: 777'

In [5]:
check_car_id('АБ22ВВ193')

'Номер не валиден'

In [6]:
check_car_id('A222BC96')

'Номер A222BC валиден. Регион: 96'

In [7]:
check_car_id('А222ВС9') 

'Номер не валиден'

In [8]:
check_car_id('А222ВС9699')

'Номер не валиден'

In [9]:
check_car_id('Я222ВС96') 

'Номер не валиден'

In [10]:
check_car_id('')

'Номер не валиден'

# Задание №2

In [11]:
def parse_logs(text: str) -> list[dict]:
    result = []
    if not text or not text.strip():
        return result
        
    strings = text.splitlines()

    pattern = r'^(?P<ts>\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\s+(?P<level>INFO|ERROR|DEBUG|WARN)\s+(?P<tail>.*)$'
    tail_pattern = r'(?P<key>user|action|ip|card)=(?P<val>\S*)'
    
    email_pattern = r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}'
    ip_pattern = r'(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)'
    card_pattern = r'\d{4}-\d{4}-\d{4}(-\d{4})'
    
    for string in strings: 
        match = re.search(pattern, string) 
        if not match: 
            continue 
        dct = match.groupdict() 
        tail = dct['tail'] 
        errors = []
        fields = {} 
        for field_match in re.finditer(tail_pattern, tail): 
            fields[field_match.group('key')] = field_match.group('val') 
        
        user = fields.get('user') 
        action = fields.get('action') 
        ip = fields.get('ip') 
        card = fields.get('card')
        if user is None: 
            errors.append('missing_user') 
        elif not re.fullmatch(email_pattern, user): 
            errors.append('invalid_email')  
        if action is None: 
            errors.append('missing_action')
        if ip is None: 
            errors.append('missing_ip') 
        elif not re.fullmatch(ip_pattern, ip): 
            errors.append('invalid_ip')
        if card is not None: 
            if re.fullmatch(card_pattern, card): 
                card = '****-****-****-' + card[-4:] 
            else: 
                errors.append('invalid_card') 
                card = None 
        result.append({ 
            'ts': dct['ts'], 
            'level': dct['level'], 
            'user': user, 
            'action': action, 
            'ip': ip, 
            'card': card, 
            'errors': errors })
    return result

In [12]:
with open('logs_hw4.txt', 'r') as file:
    text = file.read()
    print(parse_logs(text))

[{'ts': '2024-03-15 10:51:25', 'level': 'WARN', 'user': 'user+tag@gmail.com', 'action': 'login', 'ip': '203.0.113.42', 'card': None, 'errors': []}, {'ts': '2024-13-01 12:00:00', 'level': 'INFO', 'user': 'ivan@mail.ru', 'action': 'login', 'ip': '1.1.1.1', 'card': None, 'errors': []}, {'ts': '2024-03-15 11:46:57', 'level': 'DEBUG', 'user': 'petr.sidorov@yandex.ru', 'action': 'login', 'ip': '8.8.8.8', 'card': None, 'errors': []}, {'ts': '2024-03-15 09:36:54', 'level': 'WARN', 'user': 'petr.sidorov@yandex.ru', 'action': 'retry', 'ip': '999.1.1.1', 'card': None, 'errors': ['invalid_ip']}, {'ts': '2024-03-15 09:07:09', 'level': 'ERROR', 'user': 'user@domain.', 'action': 'delete', 'ip': '0.0.0.0', 'card': None, 'errors': ['invalid_email']}, {'ts': '2024-03-15 10:28:11', 'level': 'ERROR', 'user': '@onlydomain.com', 'action': 'pay', 'ip': '300.300.300.300', 'card': None, 'errors': ['invalid_email', 'invalid_ip', 'invalid_card']}, {'ts': '2024-03-15 10:04:06', 'level': 'ERROR', 'user': 'user+tag